In [1]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

# 1. Configuración de rutas
# Rutas relativas según tu estructura
input_folder = "../data/raw"
filename = "tarifa_efectiva_total_hs6_countries.xlsx"

input_path = Path(input_folder) / filename

output_folder = "../data/intermediate"
output_filename = "Efectivas_HTS_Completo.xlsx"
output_path = Path(output_folder) / output_filename

# Crear carpeta de salida si no existe
os.makedirs(output_folder, exist_ok=True)

# 2. Cargar el archivo
print(f"Leyendo archivo desde: {input_path}")
try:
    # Leemos el Excel. Por defecto pandas intenta reconocer NA, 
    # pero si están escritos como texto "NaN", a veces los carga como strings.
    df = pd.read_excel(input_path)
except ValueError:
    df = pd.read_csv(input_path)
except FileNotFoundError:
    print(f"Error: No se encontró el archivo en {input_path}")
    exit()

# 3. Limpieza y Homogeneización de NAs
# Primero aseguramos que la fecha sea datetime
df['Fecha'] = pd.to_datetime(df['Fecha'])

# Identificamos las columnas que NO son identificadores (ni Fecha ni Subpartida)
# Asumimos que el resto son columnas numéricas (Total, Mexico, China, etc.)
cols_to_clean = [col for col in df.columns if col not in ['Fecha', 'Subpartida']]

print(f"Limpiando columnas numéricas: {cols_to_clean}")

for col in cols_to_clean:
    # Esto convierte todo lo que sea número a número, y lo que sea texto (como "NaN") a np.nan (vacío real)
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Ahora todos los 'NaN' escritos son idénticos a los vacíos reales.

# 4. Definir el rango completo de fechas
min_date = df['Fecha'].min()
max_date = df['Fecha'].max()
print(f"Rango de fechas: {min_date.date()} a {max_date.date()}")

all_dates = pd.date_range(start=min_date, end=max_date, freq='MS')

# 5. Crear la estructura base (scaffold)
unique_subpartidas = df['Subpartida'].unique()

multi_index = pd.MultiIndex.from_product(
    [unique_subpartidas, all_dates], 
    names=['Subpartida', 'Fecha']
)

df_base = pd.DataFrame(index=multi_index).reset_index()

# 6. Unir con los datos originales
# Al hacer esto, las fechas nuevas tendrán np.nan automáticamente, 
# que ahora son idénticos a los que limpiamos en el paso 3.
df_merged = pd.merge(df_base, df, on=['Subpartida', 'Fecha'], how='left')

# (Opcional) Si necesitas recalcular columnas derivadas, hazlo aquí.
# Como este archivo es de Tarifas, no recalculamos porcentajes de participación 
# a menos que sea necesario.

# 7. Ordenar y guardar
df_merged = df_merged.sort_values(by=['Subpartida', 'Fecha'])

print(f"Guardando archivo en: {output_path}")
df_merged.to_excel(output_path, index=False)
print("¡Proceso terminado exitosamente! Los NAs ahora son homogéneos.")

Leyendo archivo desde: ..\data\raw\tarifa_efectiva_total_hs6_countries.xlsx
Limpiando columnas numéricas: ['Total', 'Mexico', 'China']
Rango de fechas: 2018-01-01 a 2025-10-01
Guardando archivo en: ..\data\intermediate\Efectivas_HTS_Completo.xlsx
¡Proceso terminado exitosamente! Los NAs ahora son homogéneos.
